In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, bi
from pyspark.sql import functions as F

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat = cfg["catalog"]
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if detail != "" and not ok else ""))

g = lambda t: spark.table(f"{cat}.gold.{t}")
total = lambda df, col: df.agg(F.sum(col)).first()[0] or 0
print(cat, "bi views")

In [0]:
views = {r[0] for r in spark.sql(f"""SELECT table_name FROM {cat}.information_schema.tables
                                     WHERE table_schema = 'gold' AND table_type = 'VIEW'""").collect()}
wanted = bi.wanted(cfg)
check("every BI view exists", set(wanted) <= views, sorted(set(wanted) - views))
for v in wanted:
    n = g(v).count() if v in views else 0
    check(f"{v} has rows", n > 0, n)
    print(f"INFO {v}: {n} rows")

personal = {"full_name", "phone", "aadhaar", "dob", "monthly_income", "id_fingerprint", "contact_fingerprint",
            "patient_id", "customer_id", "master_customer_id", "member_id"}
bad = {v: sorted(personal & set(g(v).columns)) for v in wanted if v in views}
check("no personal or id columns in any view", not any(bad.values()), bad)

if cfg["code"] == "bhg":
    check("v_bhg_revenue_daily total = fact_bhg_event", abs(total(g("v_bhg_revenue_daily"), "amount") - total(g("fact_bhg_event"), "amount")) < 0.01)
    check("v_group_daily total = fact_group_daily", abs(total(g("v_group_daily"), "group_amount") - total(g("fact_group_daily"), "group_amount")) < 0.01)
    check("v_group_exposure people = group_exposure", total(g("v_group_exposure"), "people") == g("group_exposure").count())
    check("v_group_customers people = persons", total(g("v_group_customers"), "people") == g("dim_group_customer").filter("group_customer_key <> -1").count())
    check("v_group_kpi = fact_group_kpi", g("v_group_kpi").count() == g("fact_group_kpi").count())
else:
    check("v_fqf_portfolio_daily total = fact_fqf_event", abs(total(g("v_fqf_portfolio_daily"), "amount") - total(g("fact_fqf_event"), "amount")) < 0.01)
    check("v_fqf_loan_book loans = dim_loan", total(g("v_fqf_loan_book"), "loans") == g("dim_loan").filter("loan_key <> -1").count())

print(f"\nt26_bi_views: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t26_bi_views failed")